In [1]:
# pip install numpy scipy

import numpy as np
from scipy.optimize import linprog, minimize, LinearConstraint

# -----------------------------
# 数据（与 MATLAB 保持一致）
# -----------------------------
a = np.array([1.25, 8.75, 0.5, 5.75, 3.0, 7.25])   # 6 个工地 x 坐标
b = np.array([1.25, 0.75, 4.75, 5.0, 6.5, 7.75])   # 6 个工地 y 坐标
x_fac_fixed = np.array([5.0, 2.0])                 # 临时料场 x 坐标（两处）
y_fac_fixed = np.array([1.0, 7.0])                 # 临时料场 y 坐标（两处）
d = np.array([3, 5, 4, 7, 6, 11], dtype=float)     # 6 个工地日用量
cap = np.array([20.0, 20.0])                       # 两个料场日储量（容量）

# ================
# 第一问：线性规划
# ================
# 计算 6 工地与两个料场的距离（共 12 个）
l = np.zeros((6, 2))
for i in range(6):
    for j in range(2):
        l[i, j] = np.hypot(x_fac_fixed[j] - a[i], y_fac_fixed[j] - b[i])

# 目标函数系数向量 f：先 6 个来自料场 1，再 6 个来自料场 2
f = np.concatenate([l[:, 0], l[:, 1]])

# 不等式约束 A_ub x <= b_ub  （两处料场的出料量不超过容量）
A_ub = np.array([
    [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]
], dtype=float)
b_ub = cap.copy()

# 等式约束 A_eq x = b_eq  （每个工地的需求被两处料场完全满足）
A_eq = np.hstack([np.eye(6), np.eye(6)])
b_eq = d.copy()

# 变量下界
bounds_lp = [(0, None)] * 12

res_lp = linprog(f, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                 bounds=bounds_lp, method="highs")

print("—— 第一问：线性规划 ——")
if res_lp.success:
    x_lp = res_lp.x
    fval_lp = res_lp.fun
    print("最优运输量分配 x（长度 12）：\n", x_lp.reshape(2, 6).T)
    print("最小总运输量（吨·千米） fval =", fval_lp)
else:
    print("求解失败：", res_lp.message)

# =================
# 第二问：非线性规划
# =================
# 变量 x 的含义：
# x[0:6]   = 料场 1 → 6 个工地的供给量
# x[6:12]  = 料场 2 → 6 个工地的供给量
# x[12:14] = 料场 1 的坐标 (x13, x14)
# x[14:16] = 料场 2 的坐标 (x15, x16)

def total_cost(x):
    # 取出坐标
    x13, x14 = x[12], x[13]  # 料场 1
    x15, x16 = x[14], x[15]  # 料场 2

    # 计算两处料场到 6 个工地的距离
    dist1 = np.hypot(x13 - a, x14 - b)
    dist2 = np.hypot(x15 - a, x16 - b)

    # 成本（吨·千米和）
    return np.dot(dist1, x[0:6]) + np.dot(dist2, x[6:12])

# 线性约束（与 MATLAB 的 A2, B2, Aeq2, beq2 对应）
# A2 x <= B2：两处料场的总出料量不超过 20
A2 = np.zeros((2, 16))
A2[0, 0:6] = 1.0
A2[1, 6:12] = 1.0
B2 = cap.copy()
lin_ub = LinearConstraint(A2, lb=-np.inf*np.ones(2), ub=B2)

# Aeq2 x = beq2：每个工地的需求由两处料场完全满足
Aeq2 = np.hstack([np.eye(6), np.eye(6), np.zeros((6, 4))])
beq2 = d.copy()
lin_eq = LinearConstraint(Aeq2, lb=beq2, ub=beq2)

# 下界（前 12 个变量 >= 0；坐标无限制）
bounds_nlp = [(0, None)] * 12 + [(None, None)] * 4

# 初始值（用第一问的分配 + 临时料场坐标）
x0 = np.zeros(16)
if res_lp.success:
    x0[0:12] = res_lp.x
else:
    x0[0:12] = np.r_[d/2, d/2]  # 兜底
x0[12:14] = [5.0, 1.0]  # 料场 1 坐标
x0[14:16] = [2.0, 7.0]  # 料场 2 坐标

res_nlp = minimize(
    total_cost, x0,
    method="SLSQP",                       # SQP 思想（与 MATLAB fmincon 'sqp' 类似）
    bounds=bounds_nlp,
    constraints=[lin_ub, lin_eq],
    options=dict(maxiter=2000, ftol=1e-9, disp=False)
)

print("\n—— 第二问：非线性规划（SLSQP） ——")
if res_nlp.success:
    x2 = res_nlp.x
    fval2 = res_nlp.fun
    print("最优解 x2（展示前 12 个配送量，后跟两处坐标）:")
    print("配送量：\n", x2[0:12].reshape(2, 6).T)
    print("料场 1 坐标：(%.6f, %.6f)" % (x2[12], x2[13]))
    print("料场 2 坐标：(%.6f, %.6f)" % (x2[14], x2[15]))
    print("最小总运输量（吨·千米） fval2 =", fval2)
else:
    print("求解失败：", res_nlp.message)

# ==========================
# 蒙特卡罗选初值（可选，加速版）
# ==========================
# 思路与 MATLAB 一致：随机生成一批可行点，取目标值更小者作为更好的初值，再交给 SLSQP 精化。
# 注意：大规模采样会很慢，这里给一个可调参数的小规模示例。

def random_feasible_sample(rng):
    """生成一个可行解样本（满足线性约束与下界）。
       - 先随机料场坐标到 [0, 9]^2（与题面坐标范围一致）
       - 再随机把每个工地的需求 d[i] 在两个料场之间分配（和为 d[i]）
       - 再检查容量约束（若超容量则缩放）
    """
    x = np.zeros(16)
    # 坐标
    x[12:14] = rng.uniform(0, 9, size=2)
    x[14:16] = rng.uniform(0, 9, size=2)
    # 每个工地把需求在两个料场间随机分配
    split = rng.random(6)  # in (0,1)
    x[0:6] = d * split
    x[6:12] = d - x[0:6]
    # 容量约束：若超过 cap，则按比例缩放
    s1 = x[0:6].sum()
    s2 = x[6:12].sum()
    if s1 > cap[0]:
        x[0:6] *= (cap[0] / s1)
        x[6:12] = d - x[0:6]
    if s2 > cap[1]:
        x[6:12] *= (cap[1] / s2)
        x[0:6] = d - x[6:12]
    # 再次确保两侧非负
    x[0:12] = np.clip(x[0:12], 0, None)
    return x

# 采样并挑选更优初值
rng = np.random.default_rng(123)
best_x0 = x0.copy()
best_val = total_cost(best_x0)

M = 20000  # 可按需调大；过大将明显变慢（MATLAB 用到了 1e6）
for _ in range(M):
    cand = random_feasible_sample(rng)
    val = total_cost(cand)
    if val < best_val:
        best_val = val
        best_x0 = cand

print("\n蒙特卡罗选取的较优初值（示例规模 M=%d）：" % M)
print("配送量：\n", best_x0[0:12].reshape(2, 6).T)
print("料场 1 坐标：(%.6f, %.6f)" % (best_x0[12], best_x0[13]))
print("料场 2 坐标：(%.6f, %.6f)" % (best_x0[14], best_x0[15]))
print("对应目标值 =", best_val)

# 用 MC 初值再次求解
res_mc = minimize(
    total_cost, best_x0,
    method="SLSQP",
    bounds=bounds_nlp,
    constraints=[lin_ub, lin_eq],
    options=dict(maxiter=4000, ftol=1e-10, disp=False)
)

print("\n—— 使用蒙特卡罗初值的精化解 ——")
if res_mc.success:
    x3 = res_mc.x
    fval3 = res_mc.fun
    print("最优解 x3（配送量 + 坐标）:")
    print("配送量：\n", x3[0:12].reshape(2, 6).T)
    print("料场 1 坐标：(%.6f, %.6f)" % (x3[12], x3[13]))
    print("料场 2 坐标：(%.6f, %.6f)" % (x3[14], x3[15]))
    print("最小总运输量（吨·千米） fval3 =", fval3)
else:
    print("求解失败：", res_mc.message)


—— 第一问：线性规划 ——
最优运输量分配 x（长度 12）：
 [[ 3.  0.]
 [ 5.  0.]
 [ 0.  4.]
 [ 7.  0.]
 [ 0.  6.]
 [ 1. 10.]]
最小总运输量（吨·千米） fval = 136.22751988318154

—— 第二问：非线性规划（SLSQP） ——
最优解 x2（展示前 12 个配送量，后跟两处坐标）:
配送量：
 [[3.00000000e+00 5.50935067e-14]
 [5.00000000e+00 4.52754244e-14]
 [4.00000000e+00 1.01126872e-12]
 [7.00000000e+00 4.67951288e-15]
 [1.00000000e+00 5.00000000e+00]
 [2.49831477e-13 1.10000000e+01]]
料场 1 坐标：(5.695926, 4.928490)
料场 2 坐标：(7.250000, 7.750000)
最小总运输量（吨·千米） fval2 = 89.88347229529086

蒙特卡罗选取的较优初值（示例规模 M=20000）：
配送量：
 [[ 0.37635078  2.62364922]
 [ 3.38384209  1.61615791]
 [ 1.07983225  2.92016775]
 [ 1.82892525  5.17107475]
 [ 0.74824073  5.25175927]
 [10.65894972  0.34105028]]
料场 1 坐标：(8.051871, 7.121326)
料场 2 坐标：(4.314756, 5.079633)
对应目标值 = 107.03041806863214

—— 使用蒙特卡罗初值的精化解 ——
最优解 x3（配送量 + 坐标）:
配送量：
 [[3.32763547e-14 3.00000000e+00]
 [5.00000000e+00 3.71578713e-14]
 [5.26378727e-14 4.00000000e+00]
 [3.58449352e-14 7.00000000e+00]
 [4.21862611e-14 6.00000000e+00]
 [1.10000000e+0